In [1]:
import torch
import numpy as np
import torch.nn as nn

In [ ]:
class UnetDownsampler(torch.nn.Module):
    def __init__(self, h, w, c, device="cuda", batch_size=32):
        super().__init__()
        self.device = device
        self.h = h
        self.w = w
        self.c = c
        self.batch_size = batch_size

        self.input = torch.nn.Conv2d(
                in_channels = c,
                out_channels = 64,
                kernel_size = (7, 7),
                stride = 1,
                padding='same' 
            ).to(device=device,   )

        self.activation = torch.nn.ReLU().to(device=device,   )
        
        self.pool_input = torch.nn.MaxPool2d(
            kernel_size=(3, 3),
            stride=4,
        ).to(device=device,   )
        self.conv_2 = torch.nn.ModuleList([
            torch.nn.Sequential(
            torch.nn.Conv2d(in_channels=64,
                            out_channels=64,
                            kernel_size=(6, 6),
                            stride=1,
                            padding='same'),
            self.activation, 
            torch.nn.Conv2d(in_channels=64,
                            out_channels=64,
                            kernel_size=(6, 6),
                            stride=1,
                            padding='same'),
            torch.nn.BatchNorm2d(64),
            ).to(device=device,   ), 
            torch.nn.Sequential(    
            torch.nn.Conv2d(in_channels=64,
                                        out_channels=64,
                                        kernel_size=(6, 6),
                                        stride=1,
                padding='same'),
                self.activation, 
                        torch.nn.Conv2d(in_channels=64,
                                        out_channels=64,
                                        kernel_size=(6, 6),
                                        stride=1,
                padding='same'),
                        torch.nn.BatchNorm2d(64),
                        ).to(device=device,   )
                    ])          
        self.reduce_dim_0 = torch.nn.Conv2d(in_channels=64, out_channels=128, 
                                            kernel_size=(1,1), stride=2).to(device=device,   )
        self.conv_3 = torch.nn.ModuleList([torch.nn.Sequential(
                    torch.nn.Conv2d(in_channels=64,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                        padding='same'),
                    self.activation, 
                    torch.nn.MaxPool2d(
                    kernel_size=(2, 2),
                    stride=2,
                    ),
                    torch.nn.Conv2d(in_channels=128,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                     padding='same'),
                     
                    torch.nn.BatchNorm2d(128),
                                 
                ).to(device=device,   ), torch.nn.Sequential(
                    torch.nn.Conv2d(in_channels=128,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                        padding='same'),
                    self.activation, 
                    torch.nn.Conv2d(in_channels=128,
                                    out_channels=128,
                                    kernel_size=(3, 3),
                                    stride=1,
                         padding='same'),
                    torch.nn.BatchNorm2d(128),
                             
                ).to(device=device,   ),
                ])
        self.reduce_dim_1 = torch.nn.Conv2d(in_channels=128, out_channels=256, 
                                                    kernel_size=(1,1), stride=2,
                                                    ).to(device=device,   )
        self.conv_4 = torch.nn.ModuleList([torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=128,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                self.activation, 
                torch.nn.MaxPool2d(
                    kernel_size=(2, 2),
                    stride=2,
                    ),
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            
                            torch.nn.BatchNorm2d(256),
                                      
                        ).to(device=device,   ), torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            self.activation, 
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=256,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.BatchNorm2d(256),
                                         
                        ).to(device=device,   ),
                        ])
        self.reduce_dim_2 = torch.nn.Conv2d(in_channels=256, out_channels=512, 
                                                            kernel_size=(1,1), stride=2).to(device=device,   )
        self.conv_5 = torch.nn.ModuleList([torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=256,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                torch.nn.MaxPool2d(
                    kernel_size=(2, 2),
                    stride=2,
                    ),
                            torch.nn.Conv2d(in_channels=512,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.BatchNorm2d(512),
                            self.activation,              
                        ).to(device=device,   ), torch.nn.Sequential(
                            torch.nn.Conv2d(in_channels=512,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            self.activation, 
                            torch.nn.Conv2d(in_channels=512,
                                            out_channels=512,
                                            kernel_size=(3, 3),
                                            stride=1,
                padding='same'),
                            torch.nn.BatchNorm2d(512),
                                       
                        ).to(device=device,   ),
                        ])
        self.average_pool = torch.nn.AvgPool2d((4, 4), 2)

    def forward(self, images):
        ## Create implementation
        res_0 = self.input(images)
        #print(res_0.shape)
        res_0 = self.pool_input(self.activation(res_0))
        #print("1")
        x_0 = self.conv_2[0](res_0) 
        #print(res_0.device, x_0.device, torch.nn.functional.pad(res_0, (0,0, 0,0, 0, x_0.shape[0]- res_0.shape[0]),mode="constant",value = 0).to(device=self.device ,   ).device)
        x_0 = self.activation(torch.nn.functional.pad(res_0, (0,0, 0,0, 0, x_0.shape[1]- res_0.shape[1]),mode="constant",value = 0).to(device=self.device ,   ) +x_0)
        
        x_1 = self.conv_2[1](x_0) 
        x_1 = self.activation(torch.nn.functional.pad(x_0, (0,0, 0,0, 0, x_1.shape[1]- x_0.shape[1]), mode="constant", value = 0).to(device=self.device ,   )+ x_1)
        #print("2"
        res_1 = self.reduce_dim_0(x_1)
        #print(res_1.shape)
        x_2 = self.conv_3[0](x_1)
        #print(x_2.shape, x_1.shape)
        x_2 = self.activation(x_2 + res_1)
        x_3 = self.conv_3[1](x_2)
        x_3 = self.activation(torch.nn.functional.pad(x_2, (0,0, 0,0, 0, x_3.shape[1]- x_2.shape[1]), mode="constant", value = 0).to(device=self.device ,   )+ x_3)
        #print("3")
        res_2 = self.reduce_dim_1(x_3)
        
        x_4 = self.activation(self.conv_4[0](x_3)+res_2)
        x_5 = self.conv_4[1](x_4)
        x_5 = self.activation(torch.nn.functional.pad(x_4, (0,0, 0,0, 0, x_5.shape[1]- x_4.shape[1]), mode="constant", value = 0).to(device=self.device ,   )+ x_5)
        #print("4")
        res_3 = self.reduce_dim_2(x_5)
        x_6 = self.activation(self.conv_5[0](x_5)+res_3)
        x_7_ = self.conv_5[1](x_6)
        x_7 = self.activation(torch.nn.functional.pad(x_6, (0,0, 0,0, 0, x_7.shape[1]- x_6.shape[1]), mode="constant", value = 0).to(device=self.device ,   )+ x_7)
        x_7 = self.average_pool(x_7)
        return x_7, x_




In [ ]:
class UnetUpSampler(nn.Module):
    def __init__(self, )